In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [14]:
selected_features = [
"hpv_related",
"oropharynx",
"uicc8_III-IV",
"cavum_oris",
"charlson"      
]

# Selecting features in the DataFrame
X_plsr = X[selected_features]
X_new = X_plsr.copy()

In [15]:
X_MAASTRO_plsr = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_plsr.copy()
# No need to standardize since all selected columns are binary
X_new_std = X_new
MAASTRO_new_std = MAASTRO_new

# Standardization

In [18]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)
"""
# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 

# All the columns are binary meaning no scaler needed 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
"""

'\n# Do the standardization for the numeric part \nscaler = RobustScaler() \nX_new_numeric_columns = X_new_numeric.columns\nX_new_numeric_index = X_new_numeric.index \n\n# All the columns are binary meaning no scaler needed \nX_new_numeric_std = scaler.fit_transform(X_new_numeric)\nX_new_numeric_std = pd.DataFrame(X_new_numeric_std,\n                                 columns=X_new_numeric_columns, \n                                 index=X_new_numeric_index)\nX_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)\n\n# Change the order of the X_new_std \nX_new_std = X_new_std[X_new.columns]\n'

In [19]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
"""
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]
"""


'\nMAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns\nMAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns\nMAASTRO_new_numeric_index = MAASTRO_new_numeric.index \nMAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)\nMAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,\n                                 columns=MAASTRO_new_numeric_columns, \n                                 index=MAASTRO_new_numeric_index)\nMAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)\n\n# Change the order of the X_new_std \nMAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]\n'

In [20]:
X_new

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,charlson
0,0.0,1,0.0,0,0
1,0.0,0,0.0,0,1
2,0.0,0,1.0,1,1
3,0.0,0,0.0,0,1
4,0.0,0,0.0,0,1
...,...,...,...,...,...
134,1.0,1,0.0,0,0
135,1.0,1,1.0,0,0
136,1.0,1,0.0,0,1
137,1.0,1,1.0,0,1


In [21]:
X_new_std

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,charlson
0,0.0,1,0.0,0,0
1,0.0,0,0.0,0,1
2,0.0,0,1.0,1,1
3,0.0,0,0.0,0,1
4,0.0,0,0.0,0,1
...,...,...,...,...,...
134,1.0,1,0.0,0,0
135,1.0,1,1.0,0,0
136,1.0,1,0.0,0,1
137,1.0,1,1.0,0,1


In [22]:
MAASTRO_new

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,charlson
0,1,1,0,0,1
1,0,1,1,0,0
2,0,1,1,0,1
3,0,0,1,0,1
4,1,1,0,0,1
...,...,...,...,...,...
94,0,0,1,0,0
95,0,0,1,0,1
96,1,1,1,0,1
97,1,1,0,0,0


In [23]:
MAASTRO_new_std

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,charlson
0,1,1,0,0,1
1,0,1,1,0,0
2,0,1,1,0,1
3,0,0,1,0,1
4,1,1,0,0,1
...,...,...,...,...,...
94,0,0,1,0,0
95,0,0,1,0,1
96,1,1,1,0,1
97,1,1,0,0,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [24]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:36:57,803] A new study created in memory with name: no-name-95f42188-ca37-4374-841d-e8df9d6430be
python(44932) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.7209821428571429
Fold 3 C-index: 0.7916666666666666


[I 2024-04-14 19:37:11,029] A new study created in memory with name: no-name-e92ebe9a-505c-4bba-9eb5-a598f79b550e


Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:11,009] Trial 0 finished with value: 0.7428869028324144 and parameters: {}. Best is trial 0 with value: 0.7428869028324144.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7428869028324144], datetime_start=datetime.datetime(2024, 4, 14, 19, 36, 58, 120433), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 11, 7860), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7428869028324144


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1559351021475166
Fold 2 IBS: 0.20199327049535848
Fold 3 IBS: 0.14988167065458602
Fold 4 IBS: 0.15052551170060235
Fold 5 IBS: 0.2607666311501383
[I 2024-04-14 19:37:11,846] Trial 0 finished with value: 0.18382043722964034 and parameters: {}. Best is trial 0 with value: 0.18382043722964034.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18382043722964034], datetime_start=datetime.datetime(2024, 4, 14, 19, 37, 11, 159313), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 11, 845925), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18382043722964034


In [25]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [26]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.184


#### Test

In [27]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [28]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.583
IBS score: 0.266


In [29]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [30]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [31]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:37:12,266] A new study created in memory with name: no-name-71bd0862-2d25-431a-b406-065ceaf72d5e


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7424242424242424
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8016877637130801


[I 2024-04-14 19:37:12,636] A new study created in memory with name: no-name-e3431652-53f2-4407-8736-41f417e0985b


Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 19:37:12,630] Trial 0 finished with value: 0.7182877718827688 and parameters: {}. Best is trial 0 with value: 0.7182877718827688.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7182877718827688], datetime_start=datetime.datetime(2024, 4, 14, 19, 37, 12, 311303), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 12, 629390), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7182877718827688


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651763328032
Fold 2 IBS: 0.2215779146949956
Fold 3 IBS: 0.20453594133219627
Fold 4 IBS: 0.22473803324306438
Fold 5 IBS: 0.2181243143630602
[I 2024-04-14 19:37:12,909] Trial 0 finished with value: 0.21659054425331936 and parameters: {}. Best is trial 0 with value: 0.21659054425331936.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054425331936], datetime_start=datetime.datetime(2024, 4, 14, 19, 37, 12, 674703), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 12, 908970), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054425331936


In [32]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [33]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.718
train_ibs:  0.217


#### Test

In [34]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [35]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.624


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [36]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [37]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:37:13,254] A new study created in memory with name: no-name-5f35a973-0264-49ad-81d2-962c1206c73b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.7209821428571429
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312


[I 2024-04-14 19:37:14,387] A new study created in memory with name: no-name-3ed29f8d-91d8-4f40-b5eb-4480d1427464


Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:14,373] Trial 0 finished with value: 0.7428869028324144 and parameters: {}. Best is trial 0 with value: 0.7428869028324144.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7428869028324144], datetime_start=datetime.datetime(2024, 4, 14, 19, 37, 13, 294769), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 14, 372877), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7428869028324144


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.15606861576698203
Fold 2 IBS: 0.20309895455300028
Fold 3 IBS: 0.14946560041623583
Fold 4 IBS: 0.14998880386341362
Fold 5 IBS: 0.25859361321320185
[I 2024-04-14 19:37:15,703] Trial 0 finished with value: 0.18344311756256673 and parameters: {}. Best is trial 0 with value: 0.18344311756256673.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18344311756256673], datetime_start=datetime.datetime(2024, 4, 14, 19, 37, 14, 587593), datetime_complete=datetime.datetime(2024, 4, 14, 19, 37, 15, 702918), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18344311756256673


In [38]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [39]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.183


#### Test 

In [40]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [41]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.583


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.261


In [42]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [43]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:37:16,309] A new study created in memory with name: no-name-b7f8ae0a-e4e9-48c7-ba9f-30a0e4739fb8


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:17,166] Trial 0 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.739315474260986.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:18,160] Trial 1 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.739315474260986.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:19,082] Trial 2 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.7393

Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:31,296] Trial 24 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:31,799] Trial 25 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:32,082] Trial 26 finished with value: 0.715225727602829 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3

Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:51,292] Trial 48 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.8342998569224302}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.7209821428571429
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:51,845] Trial 49 finished with value: 0.7428869028324144 and parameters: {'l1_ratio': 0.9586049921503356}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:37:52,475] Trial 50 finished with value: 0.739315474260986 and parameters: {'l1_ratio': 0.7809764372372684}. Best is trial 6 with value: 0.7428869028324144.
Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.72098214285714

...

Fold 1 IBS: 0.15597895981533158
Fold 2 IBS: 0.2032925960225151
Fold 3 IBS: 0.14940311636030415
Fold 4 IBS: 0.14997720075183757
Fold 5 IBS: 0.25849972609225597
[I 2024-04-14 19:38:31,277] Trial 0 finished with value: 0.18343031980844887 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18343031980844887.
Fold 1 IBS: 0.15583301273157582
Fold 2 IBS: 0.20371525425477433
Fold 3 IBS: 0.1492910084197231
Fold 4 IBS: 0.1499540199957421
Fold 5 IBS: 0.25844150966558693
[I 2024-04-14 19:38:31,979] Trial 1 finished with value: 0.18344696101348043 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18343031980844887.
Fold 1 IBS: 0.1558054908902288
Fold 2 IBS: 0.20367466951648294
Fold 3 IBS: 0.14925365786585554
Fold 4 IBS: 0.1499837506928614
Fold 5 IBS: 0.2583180749629356
[I 2024-04-14 19:38:32,724] Trial 2 finished with value: 0.18340712878567283 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.183407128785672

Fold 1 IBS: 0.15577733180113215
Fold 2 IBS: 0.20379916407629925
Fold 3 IBS: 0.14925543189757615
Fold 4 IBS: 0.14995672855687775
Fold 5 IBS: 0.25835503611862165
[I 2024-04-14 19:38:53,712] Trial 25 finished with value: 0.18342873849010138 and parameters: {'l1_ratio': 0.17588003272672872}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.15593227509866808
Fold 2 IBS: 0.2034241332673039
Fold 3 IBS: 0.1493537500026694
Fold 4 IBS: 0.14997183886083232
Fold 5 IBS: 0.258367389632833
[I 2024-04-14 19:38:54,706] Trial 26 finished with value: 0.18340987737246134 and parameters: {'l1_ratio': 0.5552319424045261}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.1557499836690634
Fold 2 IBS: 0.20378366477860374
Fold 3 IBS: 0.1492561074293218
Fold 4 IBS: 0.14998282474592273
Fold 5 IBS: 0.25840110329481447
[I 2024-04-14 19:38:55,896] Trial 27 finished with value: 0.18343473678354522 and parameters: {'l1_ratio': 0.09486155085202524}. Best is trial 20 with value: 0.183388552

Fold 5 IBS: 0.25827441995750067
[I 2024-04-14 19:39:16,914] Trial 49 finished with value: 0.18339565540683417 and parameters: {'l1_ratio': 0.12800717752404614}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.1559506098053721
Fold 2 IBS: 0.20340059121058518
Fold 3 IBS: 0.14936286253930137
Fold 4 IBS: 0.14996377611498785
Fold 5 IBS: 0.25856927039958344
[I 2024-04-14 19:39:17,671] Trial 50 finished with value: 0.18344942201396602 and parameters: {'l1_ratio': 0.6064718647459368}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.15576156555090212
Fold 2 IBS: 0.20380449209393783
Fold 3 IBS: 0.1492509630045684
Fold 4 IBS: 0.14996600991812406
Fold 5 IBS: 0.258359822482897
[I 2024-04-14 19:39:18,517] Trial 51 finished with value: 0.18342857061008588 and parameters: {'l1_ratio': 0.13419493306849467}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.15578899165132407
Fold 2 IBS: 0.20370609452211966
Fold 3 IBS: 0.1492397213464959
Fold 4 IBS: 0.1499841

Fold 1 IBS: 0.155770270853787
Fold 2 IBS: 0.2038112348603604
Fold 3 IBS: 0.14925011059126453
Fold 4 IBS: 0.1499571439080442
Fold 5 IBS: 0.2583420369748923
[I 2024-04-14 19:39:34,469] Trial 74 finished with value: 0.18342615943766968 and parameters: {'l1_ratio': 0.15972883385367564}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.1557419327917984
Fold 2 IBS: 0.20382367358834794
Fold 3 IBS: 0.14924077240178815
Fold 4 IBS: 0.1499727299623822
Fold 5 IBS: 0.2583452614865245
[I 2024-04-14 19:39:35,153] Trial 75 finished with value: 0.18342487404616822 and parameters: {'l1_ratio': 0.08437435159842996}. Best is trial 20 with value: 0.18338855229673945.
Fold 1 IBS: 0.1557574196299789
Fold 2 IBS: 0.20380442913464988
Fold 3 IBS: 0.14925020504696107
Fold 4 IBS: 0.14996898997211272
Fold 5 IBS: 0.25836304268220334
[I 2024-04-14 19:39:35,802] Trial 76 finished with value: 0.1834288172931812 and parameters: {'l1_ratio': 0.12266833298630989}. Best is trial 20 with value: 0.183388552296

Fold 5 IBS: 0.25836485244918367
[I 2024-04-14 19:39:52,534] Trial 98 finished with value: 0.1834284744395443 and parameters: {'l1_ratio': 0.09329088990333982}. Best is trial 91 with value: 0.18338016268610513.
Fold 1 IBS: 0.15575909807747473
Fold 2 IBS: 0.20382343894822838
Fold 3 IBS: 0.14924391721221436
Fold 4 IBS: 0.14996024903228422
Fold 5 IBS: 0.2583316574046228
[I 2024-04-14 19:39:53,690] Trial 99 finished with value: 0.1834236721349649 and parameters: {'l1_ratio': 0.13255336571837883}. Best is trial 91 with value: 0.18338016268610513.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.18338016268610513], datetime_start=datetime.datetime(2024, 4, 14, 19, 39, 45, 691359), datetime_complete=datetime.datetime(2024, 4, 14, 19, 39, 46, 675796), params={'l1_ratio': 0.08085767241718195}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=91, value=No

In [44]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [45]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.183


#### Test

In [46]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [47]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.583


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08085767241718195)

test_ibs:  0.261


In [48]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [49]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:39:54,306] A new study created in memory with name: no-name-9cb1aab7-4e3d-4e44-97d6-aeb4860e172d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.6785714285714286
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6056338028169014
[I 2024-04-14 19:39:59,757] Trial 0 finished with value: 0.7331704503062509 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7331704503062509.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.6808035714285714
Fold 3 C-index: 0.8112745098039216
Fold 4 C-index: 0.8080168776371308
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 19:40:03,652] Trial 1 finished with value: 0.7418310764029108 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.596244131455399
[I 2024-04-14 19:40:49,414] Trial 15 finished with value: 0.7438195723110067 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 5, 'n_estimators': 27, 'oob_score': True, 'max_samples': 0.9916308141555894, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2038091856577342, 'warm_start': True}. Best is trial 14 with value: 0.7558530252568458.
Fold 1 C-index: 0.7770562770562771
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8037974683544303
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 19:40:49,756] Trial 16 finished with value: 0.759486772674677 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.963206401785182, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3769624693898471, 'warm_start': True}. Best is trial 

Fold 5 C-index: 0.6056338028169014
[I 2024-04-14 19:41:05,978] Trial 30 finished with value: 0.7465362511720519 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 18, 'min_samples_leaf': 10, 'max_depth': 9, 'n_estimators': 180, 'oob_score': True, 'max_samples': 0.6961599111587151, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.2262327581176415, 'warm_start': True}. Best is trial 16 with value: 0.759486772674677.
Fold 1 C-index: 0.7424242424242424
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 19:41:06,529] Trial 31 finished with value: 0.7349357922961401 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 14, 'max_depth': 8, 'n_estimators': 26, 'oob_score': True, 'max_samples': 0.9351480862025267, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.45180415400387103, 'warm_start': True}. Best is trial 16 with value: 0.759486772674677.

...

Fold 5 IBS: 0.21779182387746962
[I 2024-04-14 19:44:47,008] Trial 31 finished with value: 0.21638735176568988 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 15, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 248, 'oob_score': False, 'max_samples': 0.2509419638006214, 'max_features': None, 'min_weight_fraction_leaf': 0.13797627619743888}. Best is trial 22 with value: 0.18556358157733363.
Fold 1 IBS: 0.15911475932841543
Fold 2 IBS: 0.22969020074696483
Fold 3 IBS: 0.1598944330555651
Fold 4 IBS: 0.1560912834375737
Fold 5 IBS: 0.2346999284506904
[I 2024-04-14 19:44:52,704] Trial 32 finished with value: 0.1878981210038419 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 380, 'oob_score': False, 'max_samples': 0.12343238195577959, 'max_features': None, 'min_weight_fraction_leaf': 0.03450750693525531}. Best is trial 22 with value: 0.18556358157733363.
Fold 1 IBS: 0.21394148197836613
Fold 2 IBS: 0.220677

Fold 1 IBS: 0.16258340227838988
Fold 2 IBS: 0.226804416564432
Fold 3 IBS: 0.16270125876160801
Fold 4 IBS: 0.15497094833655034
Fold 5 IBS: 0.2395972513875397
[I 2024-04-14 19:46:46,520] Trial 47 finished with value: 0.189331455465704 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 5, 'n_estimators': 325, 'oob_score': False, 'max_samples': 0.8252659010917025, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.054586859539885686}. Best is trial 22 with value: 0.18556358157733363.
Fold 1 IBS: 0.21399426337349062
Fold 2 IBS: 0.22211586414919254
Fold 3 IBS: 0.2044981281283501
Fold 4 IBS: 0.22515630305409148
Fold 5 IBS: 0.2177243240610341
[I 2024-04-14 19:46:47,748] Trial 48 finished with value: 0.21669777655323177 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 8, 'min_samples_leaf': 15, 'max_depth': 15, 'n_estimators': 40, 'oob_score': False, 'max_samples': 0.3940370420761242, 'max_features': 'log2', 'min_weight_fraction_lea

Fold 1 IBS: 0.2139502037041144
Fold 2 IBS: 0.2205253410474416
Fold 3 IBS: 0.20539138240116334
Fold 4 IBS: 0.22483265303988334
Fold 5 IBS: 0.21729817764200712
[I 2024-04-14 19:48:07,885] Trial 63 finished with value: 0.21639955156692198 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 330, 'oob_score': False, 'max_samples': 0.14304382305903557, 'max_features': None, 'min_weight_fraction_leaf': 0.212666616348277}. Best is trial 22 with value: 0.18556358157733363.
Fold 1 IBS: 0.16044995955528968
Fold 2 IBS: 0.23495388944522017
Fold 3 IBS: 0.15877526237480574
Fold 4 IBS: 0.15187995834572193
Fold 5 IBS: 0.2443346149864615
[I 2024-04-14 19:48:09,775] Trial 64 finished with value: 0.19007873694149982 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 19, 'n_estimators': 103, 'oob_score': False, 'max_samples': 0.1750738410149462, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.16122065792349757
Fold 2 IBS: 0.21325686305320374
Fold 3 IBS: 0.15822275060218324
Fold 4 IBS: 0.14281519984729882
Fold 5 IBS: 0.2458841078408022
[I 2024-04-14 19:48:53,905] Trial 79 finished with value: 0.18427991585339712 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.2969951845956973, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.01531063774064334}. Best is trial 76 with value: 0.18088658194986512.
Fold 1 IBS: 0.16590885354349483
Fold 2 IBS: 0.20627975780151778
Fold 3 IBS: 0.15178945005683334
Fold 4 IBS: 0.14741838302275773
Fold 5 IBS: 0.24237922921628863
[I 2024-04-14 19:48:54,991] Trial 80 finished with value: 0.18275513472817845 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 36, 'oob_score': True, 'max_samples': 0.29124216175839035, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.16758785531637785
Fold 2 IBS: 0.19991680056926603
Fold 3 IBS: 0.16807972276243374
Fold 4 IBS: 0.16678300610940044
Fold 5 IBS: 0.23011961493856892
[I 2024-04-14 19:49:09,871] Trial 95 finished with value: 0.18649739993920939 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 13, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 40, 'oob_score': True, 'max_samples': 0.3130463719118361, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07417173322008858}. Best is trial 76 with value: 0.18088658194986512.
Fold 1 IBS: 0.16071713586864783
Fold 2 IBS: 0.2134228435221808
Fold 3 IBS: 0.15838796300027969
Fold 4 IBS: 0.14827158241140817
Fold 5 IBS: 0.24299630373240863
[I 2024-04-14 19:49:11,528] Trial 96 finished with value: 0.18475916570698503 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 85, 'oob_score': True, 'max_samples': 0.3412497541587062, 'max_features': 'auto', 'min_weight_fraction_lea

In [50]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [51]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.77
train_ibs:  0.181


#### Test

In [52]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [53]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_features='auto', max_leaf_nodes=10,
                     max_samples=0.939974221909062, min_samples_leaf=1,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.010097497494874971,
                     n_estimators=393, random_state=123, warm_start=True)

test_cindex:  0.579


RandomSurvivalForest(max_depth=20, max_features='auto', max_leaf_nodes=12,
                     max_samples=0.22393267024094543, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.017121900234687792,
                     n_estimators=88, oob_score=True, random_state=123)

test_ibs:  0.226


In [54]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [55]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:49:16,413] A new study created in memory with name: no-name-5ef5a1ca-2bc5-4309-af72-8bc22eaa320b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.7611607142857143
Fold 3 C-index: 0.8112745098039216
Fold 4 C-index: 0.8080168776371308
Fold 5 C-index: 0.647887323943662
[I 2024-04-14 19:49:17,752] Trial 0 finished with value: 0.7597804392466398 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7597804392466398.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:49:21,005] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 19:50:00,964] Trial 16 finished with value: 0.7529245811519311 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 330, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.18687423911629308}. Best is trial 6 with value: 0.7606462401124408.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:50:01,959] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 194, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.5688392530311949, 'min_weight_fraction_leaf': 0.386240377400086

Fold 1 C-index: 0.7792207792207793
Fold 2 C-index: 0.7611607142857143
Fold 3 C-index: 0.8112745098039216
Fold 4 C-index: 0.8080168776371308
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 19:50:31,311] Trial 31 finished with value: 0.7596341067059411 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 9, 'min_samples_leaf': 5, 'max_depth': 10, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6970733166357045, 'min_weight_fraction_leaf': 0.07324922325860758}. Best is trial 6 with value: 0.7606462401124408.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6009389671361502
[I 2024-04-14 19:50:33,079] Trial 32 finished with value: 0.7438115697501873 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 469, 'oob_score': False, 'warm_start': True, 'max_featur

...

Fold 1 IBS: 0.2042662673434637
Fold 2 IBS: 0.21546039940178915
Fold 3 IBS: 0.19877151930321105
Fold 4 IBS: 0.218842890661761
Fold 5 IBS: 0.21704441300512034
[I 2024-04-14 19:54:47,085] Trial 30 finished with value: 0.21087709794306903 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 3, 'n_estimators': 225, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.36648983755402775, 'min_weight_fraction_leaf': 0.17141436902843765}. Best is trial 23 with value: 0.18723967024222346.
Fold 1 IBS: 0.1727569377514147
Fold 2 IBS: 0.20374901180658242
Fold 3 IBS: 0.1692787734812299
Fold 4 IBS: 0.17063432820059057
Fold 5 IBS: 0.22515507125242631
[I 2024-04-14 19:54:49,430] Trial 31 finished with value: 0.18831482449844877 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 15, 'min_samples_leaf': 12, 'max_depth': 7, 'n_estimators': 160, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.7936

Fold 1 IBS: 0.16460134502827933
Fold 2 IBS: 0.21567665564125096
Fold 3 IBS: 0.15786962924361955
Fold 4 IBS: 0.1457081538984672
Fold 5 IBS: 0.2517810541892506
[I 2024-04-14 19:55:18,150] Trial 45 finished with value: 0.18712736760017354 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 34, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8844796228624537, 'min_weight_fraction_leaf': 0.024268316111900768}. Best is trial 32 with value: 0.1870813475489476.
Fold 1 IBS: 0.17197178823708986
Fold 2 IBS: 0.19954592203746543
Fold 3 IBS: 0.17248894130385498
Fold 4 IBS: 0.17214941944677983
Fold 5 IBS: 0.22837206144180622
[I 2024-04-14 19:55:19,140] Trial 46 finished with value: 0.18890562649339926 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 2, 'n_estimators': 39, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.79920341

Fold 1 IBS: 0.1721755182324898
Fold 2 IBS: 0.20424076197040372
Fold 3 IBS: 0.17136470654034283
Fold 4 IBS: 0.1728480603669238
Fold 5 IBS: 0.22380784968166043
[I 2024-04-14 19:55:43,075] Trial 60 finished with value: 0.18888737935836414 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 13, 'min_samples_leaf': 9, 'max_depth': 13, 'n_estimators': 144, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8749347745790171, 'min_weight_fraction_leaf': 0.1938967869214744}. Best is trial 32 with value: 0.1870813475489476.
Fold 1 IBS: 0.16566184498778544
Fold 2 IBS: 0.21468910294737692
Fold 3 IBS: 0.16148400541808713
Fold 4 IBS: 0.15479932358755302
Fold 5 IBS: 0.24070319617381242
[I 2024-04-14 19:55:47,032] Trial 61 finished with value: 0.18746749462292298 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 6, 'n_estimators': 275, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.996093391

Fold 1 IBS: 0.1695148548794243
Fold 2 IBS: 0.20771576818012474
Fold 3 IBS: 0.17158831703903973
Fold 4 IBS: 0.17124188587680744
Fold 5 IBS: 0.22438039278811733
[I 2024-04-14 19:56:54,984] Trial 75 finished with value: 0.18888824375270272 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 2, 'n_estimators': 401, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.23979984596241344, 'min_weight_fraction_leaf': 0.013866964013846085}. Best is trial 72 with value: 0.18325598914351482.
Fold 1 IBS: 0.1594524165745049
Fold 2 IBS: 0.20755017478644808
Fold 3 IBS: 0.16071222223169898
Fold 4 IBS: 0.15521955079153346
Fold 5 IBS: 0.23506958218150567
[I 2024-04-14 19:56:58,722] Trial 76 finished with value: 0.18360078931313822 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 2, 'n_estimators': 256, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.8241

Fold 1 IBS: 0.16101381323581304
Fold 2 IBS: 0.22142678208246622
Fold 3 IBS: 0.16844591442133142
Fold 4 IBS: 0.16830133032697386
Fold 5 IBS: 0.22922568736706048
[I 2024-04-14 19:58:14,733] Trial 90 finished with value: 0.189682705486729 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 3, 'max_depth': 1, 'n_estimators': 349, 'oob_score': False, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.910485345327682, 'min_weight_fraction_leaf': 0.062300725609488405}. Best is trial 72 with value: 0.18325598914351482.
Fold 1 IBS: 0.16292721447506833
Fold 2 IBS: 0.21021704812030823
Fold 3 IBS: 0.16179671957605893
Fold 4 IBS: 0.15591862948847224
Fold 5 IBS: 0.23423617657961798
[I 2024-04-14 19:58:19,410] Trial 91 finished with value: 0.18501915764790514 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 15, 'min_samples_leaf': 5, 'max_depth': 2, 'n_estimators': 363, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.8

In [56]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [57]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.761
train_ibs:  0.183


#### Test

In [58]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [59]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=14, max_features='log2', max_leaf_nodes=4,
                   max_samples=0.5693797534575991, min_samples_leaf=7,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.001344032287160346,
                   n_estimators=424, random_state=123, warm_start=True)

C-index score: 0.608


ExtraSurvivalTrees(max_depth=2, max_features=1, max_leaf_nodes=14,
                   max_samples=0.9327683828924904, min_samples_split=18,
                   min_weight_fraction_leaf=0.0006993958841116053,
                   n_estimators=323, random_state=123)

IBS: 0.213


In [60]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [61]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 19:59:06,441] A new study created in memory with name: no-name-3a814310-8b86-43e7-94fa-2e401bf26b2b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:59:36,946] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 19:59:51,314] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:07:34,998] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7251347360205367.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:08:32,296] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 20:19:13,086] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7251347360205367.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5558035714285714
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 20:20:20,737] Trial 26 finished with value: 0.5351043762575453 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 21:36:18,698] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 21:37:07,219] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.098509220

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 21:48:52,676] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.008782667792366316, 'dropout_rate': 0.12972298735589705, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.7134499429690735, 'min_weight_fraction_leaf': 0.19311178079767077, 'max_features': 'log2', 'min_impurity_decrease': 1.142816958470248e-06, 'validation_fraction': 0.9729459657934212, 'min_samples_split': 18, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 21:49:49,028] Trial 67 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8640185330574577, 'learning_rate': 0.0159973470

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 21:59:52,179] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527746876723795, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.2556920821676466, 'n_estimators': 442, 'criterion': 'squared_error', 'ccp_alpha': 0.5896250392603071, 'min_weight_fraction_leaf': 0.22988050749684658, 'max_features': 1, 'min_impurity_decrease': 1.4995945809430595e-05, 'validation_fraction': 0.8950354974332698, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 22:01:07,205] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7443996478445325, 'l

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 22:11:06,904] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8800231052433968, 'learning_rate': 0.015424283294721588, 'dropout_rate': 0.20093805843087115, 'n_estimators': 390, 'criterion': 'squared_error', 'ccp_alpha': 1.351539840282731, 'min_weight_fraction_leaf': 0.28958635214901884, 'max_features': 'auto', 'min_impurity_decrease': 2.329078771156988e-07, 'validation_fraction': 0.939344740485312, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 4}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 22:11:59,117] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9819209167757807, 

Fold 1 IBS: 0.21238785610218208
Fold 2 IBS: 0.22059792873287004
Fold 3 IBS: 0.203108199891751
Fold 4 IBS: 0.22281747983911723
Fold 5 IBS: 0.21781150295379875
[I 2024-04-14 22:23:31,653] Trial 99 finished with value: 0.21534459350394383 and parameters: {'subsample': 0.24286963781937687, 'learning_rate': 0.01358129947690588, 'dropout_rate': 0.25496540816023416, 'n_estimators': 454, 'criterion': 'squared_error', 'ccp_alpha': 0.004367160733265784, 'min_weight_fraction_leaf': 0.2624099419664444, 'max_features': 'auto', 'min_impurity_decrease': 5.321953542697131e-07, 'validation_fraction': 0.9620562032012763, 'min_samples_split': 11, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 1}. Best is trial 53 with value: 0.21504986372331816.


* Best trial for IBS: 
 FrozenTrial(number=53, state=TrialState.COMPLETE, values=[0.21504986372331816], datetime_start=datetime.datetime(2024, 4, 14, 21, 33, 28, 630810), datetime_complete=datetime.datetime(2024, 4, 14, 21, 34, 28, 558478), params={

In [62]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.741
train_ibs:  0.215


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.59


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009625013743012712,
                                 criterion='squared_error',
                                 dropout_rate=0.1897783294507234,
                                 learning_rate=0.015420772490455037,
                                 max_features='auto', max_leaf_nodes=14,
                                 min_impurity_decrease=5.869825897765074e-07,
                                 min_samples_leaf=13, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29880213170319914,
                                 n_estimators=430, random_state=123,
                                 subsample=0.9101431135071837,
                                 validation_fraction=0.9964423942006735)

IBS: 0.22


In [66]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 22:23:45,506] A new study created in memory with name: no-name-bb7530df-fceb-4e0a-b6ac-edbcf8ab2de0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 22:23:46,956] Trial 0 finished with value: 0.6849872959240584 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6849872959240584.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 22:23:54,379] Trial 1 finished with value: 0.6943010214142545 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6943010214142545.
Fold 1 C-index: 0.7251082251082251
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784


Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.8112745098039216
Fold 4 C-index: 0.8291139240506329
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 22:24:50,457] Trial 19 finished with value: 0.7458931019894951 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 11 with value: 0.7476247037210969.
Fold 1 C-index: 0.7748917748917749
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 22:24:57,001] Trial 20 finished with value: 0.7369130538891191 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 11 with value: 0.7476247037210969.
Fold 1 C-index: 0.7748917748917749
Fold 2 C-index: 0.703125
Fold 3 C-index: 0.8112745098039216
Fold 4 C-index: 0.8

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 22:26:18,939] Trial 38 finished with value: 0.7387528807289458 and parameters: {'subsample': 0.3297325755614151, 'dropout_rate': 0.8867789132968811, 'n_estimators': 392, 'learning_rate': 0.08480908032505553}. Best is trial 11 with value: 0.7476247037210969.
Fold 1 C-index: 0.7835497835497836
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 22:26:20,527] Trial 39 finished with value: 0.7386446556207208 and parameters: {'subsample': 0.1910828860959901, 'dropout_rate': 0.29848665859072016, 'n_estimators': 145, 'learning_rate': 0.06892741183938003}. Best is trial 11 with value: 0.7476247037210969.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7916666666666666


...

Fold 3 IBS: 0.1783999568257794
Fold 4 IBS: 0.20099313441590488
Fold 5 IBS: 0.2127622739815413
[I 2024-04-14 22:31:22,850] Trial 19 finished with value: 0.19673918994409964 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19673918994409964.
Fold 1 IBS: 0.1881299300179791
Fold 2 IBS: 0.20500626401911293
Fold 3 IBS: 0.1819421294378562
Fold 4 IBS: 0.20497765873828705
Fold 5 IBS: 0.21298081915188694
[I 2024-04-14 22:31:23,138] Trial 20 finished with value: 0.19860736027302442 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19673918994409964.
Fold 1 IBS: 0.19533105622634891
Fold 2 IBS: 0.20393569161683953
Fold 3 IBS: 0.18712698518091442
Fold 4 IBS: 0.20944879242499856
Fold 5 IBS: 0.21303933620208118
[I 2024-04-14 22:31:23,365] Trial 21 fini

Fold 3 IBS: 0.1675187073192873
Fold 4 IBS: 0.2315629911368222
Fold 5 IBS: 0.2332838201025736
[I 2024-04-14 22:31:38,710] Trial 38 finished with value: 0.20680697189300318 and parameters: {'subsample': 0.1710779175611994, 'dropout_rate': 0.9288534916051375, 'n_estimators': 197, 'learning_rate': 0.036023800474466655}. Best is trial 30 with value: 0.19587500032559918.
Fold 1 IBS: 0.1876140504745091
Fold 2 IBS: 0.20563381823568802
Fold 3 IBS: 0.17202903977012926
Fold 4 IBS: 0.2067894022485534
Fold 5 IBS: 0.21264550977192898
[I 2024-04-14 22:31:39,050] Trial 39 finished with value: 0.19694236410016178 and parameters: {'subsample': 0.3537909584345821, 'dropout_rate': 0.8744094300062906, 'n_estimators': 59, 'learning_rate': 0.04711870419619728}. Best is trial 30 with value: 0.19587500032559918.
Fold 1 IBS: 0.18448495606229867
Fold 2 IBS: 0.2155619099375562
Fold 3 IBS: 0.16824012001923247
Fold 4 IBS: 0.21346674211873537
Fold 5 IBS: 0.21402742087944407
[I 2024-04-14 22:31:39,463] Trial 40 finis

Fold 3 IBS: 0.17328317089014492
Fold 4 IBS: 0.31394529707721386
Fold 5 IBS: 0.2667682961494491
[I 2024-04-14 22:31:48,747] Trial 57 finished with value: 0.246586139989994 and parameters: {'subsample': 0.32826887806420246, 'dropout_rate': 0.9933561701936838, 'n_estimators': 283, 'learning_rate': 0.046115905841529976}. Best is trial 30 with value: 0.19587500032559918.
Fold 1 IBS: 0.20860918635569756
Fold 2 IBS: 0.21590342208816954
Fold 3 IBS: 0.19882549664838997
Fold 4 IBS: 0.22138259267967494
Fold 5 IBS: 0.21576315984193745
[I 2024-04-14 22:31:48,995] Trial 58 finished with value: 0.2120967715227739 and parameters: {'subsample': 0.5286310530837229, 'dropout_rate': 0.922313337459929, 'n_estimators': 41, 'learning_rate': 0.009368892992001188}. Best is trial 30 with value: 0.19587500032559918.
Fold 1 IBS: 0.1663205709204324
Fold 2 IBS: 0.20742615729931974
Fold 3 IBS: 0.1673944010677237
Fold 4 IBS: 0.19548723363098827
Fold 5 IBS: 0.2186581694007046
[I 2024-04-14 22:31:49,522] Trial 59 finis

Fold 3 IBS: 0.17533541667082073
Fold 4 IBS: 0.199151824633538
Fold 5 IBS: 0.2141302634338856
[I 2024-04-14 22:32:10,181] Trial 76 finished with value: 0.19358230602975196 and parameters: {'subsample': 0.13636686599390602, 'dropout_rate': 0.34829268282242476, 'n_estimators': 184, 'learning_rate': 0.015139043821359926}. Best is trial 64 with value: 0.1902159368111667.
Fold 1 IBS: 0.18026729871669211
Fold 2 IBS: 0.20071841061297166
Fold 3 IBS: 0.1764058988174647
Fold 4 IBS: 0.19883957499746607
Fold 5 IBS: 0.21357839204133217
[I 2024-04-14 22:32:12,940] Trial 77 finished with value: 0.19396191503718535 and parameters: {'subsample': 0.10015980621116063, 'dropout_rate': 0.2813897195542503, 'n_estimators': 233, 'learning_rate': 0.01050391580021415}. Best is trial 64 with value: 0.1902159368111667.
Fold 1 IBS: 0.18544854409101208
Fold 2 IBS: 0.20152802428953787
Fold 3 IBS: 0.17908634512845192
Fold 4 IBS: 0.20415170947503194
Fold 5 IBS: 0.21252533091673242
[I 2024-04-14 22:32:17,929] Trial 78 f

Fold 2 IBS: 0.21889413864524473
Fold 3 IBS: 0.1662261526779113
Fold 4 IBS: 0.20442135797401353
Fold 5 IBS: 0.22263086768420076
[I 2024-04-14 22:36:16,779] Trial 95 finished with value: 0.19437844482628436 and parameters: {'subsample': 0.17622533543010202, 'dropout_rate': 0.28294415276733026, 'n_estimators': 203, 'learning_rate': 0.025913269794610564}. Best is trial 64 with value: 0.1902159368111667.
Fold 1 IBS: 0.15207446196056826
Fold 2 IBS: 0.2109806041455053
Fold 3 IBS: 0.16768998617571493
Fold 4 IBS: 0.20401212700679955
Fold 5 IBS: 0.22643513205900653
[I 2024-04-14 22:36:18,139] Trial 96 finished with value: 0.19223846226951893 and parameters: {'subsample': 0.10119718770512731, 'dropout_rate': 0.19203824608920522, 'n_estimators': 258, 'learning_rate': 0.022376583667374196}. Best is trial 64 with value: 0.1902159368111667.
Fold 1 IBS: 0.14951781740699113
Fold 2 IBS: 0.21438676990941416
Fold 3 IBS: 0.16787377142612836
Fold 4 IBS: 0.206298769650207
Fold 5 IBS: 0.22782687152412864
[I 2

In [68]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.749
train_ibs:  0.19


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.1304931000922671,
                                              learning_rate=0.00524351186486742,
                                              n_estimators=359,
                                              random_state=123,
                                              subsample=0.13799461538888846)

C-index score: 0.596


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9681316166382135,
                                              learning_rate=0.030226818337871844,
                                              n_estimators=131,
                                              random_state=123,
                                              subsample=0.106249473229961)

IBS: 0.204


In [72]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [73]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.770,1.0
ExtraSurvivalTrees,0.761,2.0
ComponentwiseGradientBoosting,0.749,3.0
CoxPH,0.743,5.0
CoxLasso,0.743,5.0
CoxElastic,0.743,5.0
GradientBoosting,0.741,7.0
CoxRidge,0.718,8.0


In [74]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.181,1.0
CoxLasso,0.183,3.0
CoxElastic,0.183,3.0
ExtraSurvivalTrees,0.183,3.0
CoxPH,0.184,5.0
ComponentwiseGradientBoosting,0.190,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [75]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.624,1.0
ExtraSurvivalTrees,0.608,2.0
ComponentwiseGradientBoosting,0.596,3.0
GradientBoosting,0.590,4.0
CoxPH,0.583,6.0
CoxLasso,0.583,6.0
CoxElastic,0.583,6.0
Randomsurvivalforest,0.579,8.0


In [76]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.204,1.0
ExtraSurvivalTrees,0.213,2.0
GradientBoosting,0.220,3.0
CoxRidge,0.221,4.0
Randomsurvivalforest,0.226,5.0
CoxLasso,0.261,6.5
CoxElastic,0.261,6.5
CoxPH,0.266,8.0


In [77]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/robust/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_robust_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [78]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
